README!

The following directories will be setup in the directory where code executes:<br>
a. papers where all pdf papers for summarization and Q/A is used<br>
b. results where all results and metrics will be stored<br>
c. reference where the json files for the "golden" answer for Question-Answer and summarization is stored.<br>
d. models where the LLM models for evaluation and later if used, the model for judging will be stored.<br>
e. code assumes your secret keys are stored in an .env file. The default .env file name is "AI6130.env".  if not found it will ask for you to manually enter the secret keys<br>
f. To install faiss on local machine please enter in anaconda terminal :condal install -c pytorch -c nvidia faiss-gpu

In [ ]:
#Run this once in local or on colab
!huggingface-cli download meta-llama/Llama-3.2-3B --local-dir Llama-3.2-3B # remove if already downloaded the model from HuggingFace
!huggingface-cli download google/gemma-3-4B-it --local-dir gemma-3-4B-it
!pip install -U langchain langchain-huggingface transformers sentence-transformers
!pip install langchain-community
!pip install transformers
!pip install python-dotenv
!pip install huggingface_hub
!pip install langchain-huggingface
!pip install ragas langchain transformers huggingface_hub pandas torch
!pip install --upgrade ragas
!pip install bert-score rouge nltk evaluate datasets
!pip install accelerate bitsandbytes
!pip install pypdf
#need to instal

In [35]:
import os
import sys
import torch
import gc
import logging
import random
import json
import pandas as pd
import numpy as np
import re
import time
import warnings
import faiss
from typing import List, Dict, Any, Tuple, Optional
from pathlib import Path
from dotenv import load_dotenv
from glob import glob
from sentence_transformers import SentenceTransformer
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

# New imports for evaluation metrics
import evaluate
from rouge import Rouge
from bert_score import score as bert_score
import nltk
from nltk.tokenize import word_tokenize

# For RAGAS evaluation
from ragas.metrics import (
    context_precision, 
    context_recall, 
    faithfulness, 
    answer_relevancy,
    answer_correctness
)
from ragas import evaluate as ragas_evaluate
from datasets import Dataset

# Optional for open LLM judge
from transformers import pipeline

# Download NLTK data
nltk.download('punkt', quiet=True)


True

In [37]:
# Define a class for our RAG system
class RAGSystem:
    def __init__(self, embedding_model_name="sentence-transformers/all-MiniLM-L6-v2"):
        """Initialize the RAG system with embedding model and vector store"""
        self.embedding_model = SentenceTransformer(embedding_model_name)
        self.dimension = self.embedding_model.get_sentence_embedding_dimension()
        self.index = None
        self.documents = []
        self.chunk_size = 512
        self.chunk_overlap = 128
        self.logger = logging.getLogger(__name__)
        self.logger.info(f"Initialized RAG System with embedding dimension: {self.dimension}")
    
    def chunk_text(self, text: str) -> List[str]:
        """Split text into chunks of specified size with overlap"""
        # Simple sentence-based chunking
        sentences = re.split(r'(?<=[.!?])\s+', text)
        chunks = []
        current_chunk = []
        current_size = 0
        
        for sentence in sentences:
            sentence_size = len(sentence.split())
            if current_size + sentence_size <= self.chunk_size:
                current_chunk.append(sentence)
                current_size += sentence_size
            else:
                if current_chunk:
                    chunks.append(" ".join(current_chunk))
                
                # Start a new chunk, including overlap from the previous chunk
                if len(current_chunk) > 0:
                    # Calculate how many sentences to keep for overlap
                    overlap_size = 0
                    overlap_sentences = []
                    for s in reversed(current_chunk):
                        s_size = len(s.split())
                        if overlap_size + s_size <= self.chunk_overlap:
                            overlap_sentences.insert(0, s)
                            overlap_size += s_size
                        else:
                            break
                    
                    current_chunk = overlap_sentences
                    current_size = overlap_size
                else:
                    current_chunk = []
                    current_size = 0
                
                current_chunk.append(sentence)
                current_size += sentence_size
        
        # Add the last chunk if it exists
        if current_chunk:
            chunks.append(" ".join(current_chunk))
        
        return chunks
    
    def add_document(self, document: str, metadata: Optional[Dict] = None):
        """
        Process document, chunk it, and add to vector store
        """
        chunks = self.chunk_text(document)
        self.logger.info(f"Document chunked into {len(chunks)} parts")
        
        # Store document chunks and their metadata
        start_idx = len(self.documents)
        for i, chunk in enumerate(chunks):
            chunk_metadata = {
                "doc_id": start_idx + i,
                "chunk_id": i,
                "text": chunk
            }
            if metadata:
                chunk_metadata.update(metadata)
            self.documents.append(chunk_metadata)
        
        # Create or update the FAISS index
        self._update_index()
        return len(chunks)
    
    def _update_index(self):
        """Update or create the FAISS index with all document embeddings"""
        embeddings = self._get_embeddings([doc["text"] for doc in self.documents])
        
        # Create a new index if one doesn't exist
        if self.index is None:
            self.index = faiss.IndexFlatL2(self.dimension)
        else:
            # Reset the index
            self.index = faiss.IndexFlatL2(self.dimension)
        
        # Add all embeddings to the index
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)
        self.logger.info(f"Updated FAISS index with {len(self.documents)} chunks")
    
    def _get_embeddings(self, texts: List[str]) -> np.ndarray:
        #Generate embeddings for a list of texts
        return self.embedding_model.encode(texts, convert_to_numpy=True)
    
    def search(self, query: str, top_k: int = 3) -> List[Dict]:
        
        #Search for the most relevant chunks based on query.Returns a list of documents with their similarity scores
        
        if not self.documents or self.index is None:
            self.logger.warning("No documents indexed yet")
            return []
        
        # Generate query embedding
        query_embedding = self.embedding_model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_embedding)
        
        # Search the index
        distances, indices = self.index.search(query_embedding, min(top_k, len(self.documents)))
        
        # Prepare results
        results = []
        for i, doc_idx in enumerate(indices[0]):
            if doc_idx < 0 or doc_idx >= len(self.documents):
                continue
                
            # Convert L2 distance to similarity score (1 - normalized distance)
            similarity = 1.0 - min(distances[0][i] / 2.0, 1.0)
            
            doc = self.documents[doc_idx].copy()
            doc["similarity"] = similarity
            results.append(doc)
        
        # Sort by similarity (highest first)
        results.sort(key=lambda x: x["similarity"], reverse=True)
        return results

    def get_relevant_context(self, query: str, top_k: int = 3) -> str:
        #Retrieve relevant context for a query and returns a single text with all relevant chunks
     
        results = self.search(query, top_k)
        
        if not results:
            return ""
        
        # Combine all retrieved chunks
        contexts = [doc["text"] for doc in results]
        return "\n\n".join(contexts)

# Function to modify the existing code to use RAG
def setup_rag_system(paper_contents: Dict[str, str]) -> RAGSystem:
    
    #Set up the RAG system by processing all paper contents
    #Returns the initialized RAG system
    
    logger = logging.getLogger(__name__)
    rag = RAGSystem()
    
    for paper_file, content in paper_contents.items():
        logger.info(f"Adding document to RAG system: {paper_file}")
        rag.add_document(content, metadata={"source": paper_file})
    
    logger.info(f"RAG system initialized with {len(rag.documents)} chunks from {len(paper_contents)} documents")
    return rag

In [38]:
#Load data here
def load_document(file_path):
    logger.info(f"Loading document: {file_path}")
    try:
        if file_path.endswith('.pdf'):
            loader = PyPDFLoader(file_path)
            return loader.load()
        elif file_path.endswith(('.txt', '.md')):
            loader = TextLoader(file_path)
            return loader.load()
        else:
            logger.error(f"Unsupported file type: {file_path}")
            return []
    except Exception as e:
        logger.error(f"Error loading file {file_path}: {str(e)}")
        return []

def load_pdf_documents(papers_dir: str) -> Dict[str, str]:
    """
    Load multiple PDF documents from papers directory
    Returns a dictionary mapping filename to content
    """
    paper_files = [f for f in os.listdir(papers_dir) if f.endswith('.pdf')]
    if not paper_files:
        logger.error(f"No PDF files found in {papers_dir}")
        return {}
    
    paper_contents = {}
    for paper_file in paper_files:
        paper_path = os.path.join(papers_dir, paper_file)
        logger.info(f"Loading document: {paper_path}")
        try:
            doc_sections = load_document(paper_path)
            if doc_sections:
                content = "\n\n".join([section.page_content for section in doc_sections])
                paper_contents[paper_file] = content
                logger.info(f"Successfully loaded {paper_file} ({len(content)} characters)")
            else:
                logger.warning(f"No content loaded from {paper_file}")
        except Exception as e:
            logger.error(f"Error loading file {paper_file}: {str(e)}")
    
    return paper_contents

def load_reference_data(reference_dir: str) -> Tuple[Dict[str, List[Dict]], Dict[str, str]]:
    """
    Load reference QA pairs and summaries from the reference directory
    Returns a tuple of (qa_pairs_dict, summaries_dict)
    """
    qa_pairs_dict = {}
    summaries_dict = {}
    
    # Load QA pairs
    qa_file = os.path.join(reference_dir, "qa_pairs.json")
    if os.path.exists(qa_file):
        try:
            with open(qa_file, "r") as f:
                qa_pairs_dict = json.load(f)
            logger.info(f"Loaded QA pairs for {len(qa_pairs_dict)} papers")
        except Exception as e:
            logger.error(f"Error loading QA pairs: {str(e)}")
    else:
        logger.warning(f"QA pairs file not found at {qa_file}")
    
    # Load summaries
    summary_files = glob(os.path.join(reference_dir, "*_summary.json"))
    for summary_file in summary_files:
        try:
            paper_name = os.path.basename(summary_file).replace("_summary.json", "")
            with open(summary_file, "r") as f:
                summary_data = json.load(f)
            summaries_dict[paper_name] = summary_data.get("summary", "")
            logger.info(f"Loaded summary for {paper_name}")
        except Exception as e:
            logger.error(f"Error loading summary from {summary_file}: {str(e)}")
    
    return qa_pairs_dict, summaries_dict

In [39]:
def generate_answer_with_rag(model, tokenizer, rag_system, question, device, seed=None, model_name=""):

    #Generate an answer to a question using RAG to retrieve relevant context first

    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # Get relevant context from RAG
    retrieved_context = rag_system.get_relevant_context(question, top_k=3)
    
    # Create the system prompt
    system_prompt = """You are a helpful research assistant that answers questions based on academic papers. 
You should provide concise, accurate answers based solely on the information in the paper context provided."""

    # Create user prompt with the retrieved context
    user_prompt = f"""Based on the following academic paper excerpts, please answer the question:

PAPER CONTENT:
{retrieved_context}

QUESTION:
{question}

Please provide a direct, concise answer using only information from the paper. If the paper doesn't contain relevant information to answer the question, please state that."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
"""
    
    # Log the prompt
    logger = logging.getLogger(__name__)
    logger.info(f"Generating answer using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=500,  # Shorter for answers
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

# Modified function to use RAG for summarization
def generate_summary_with_rag(model, tokenizer, rag_system, paper_content, device, seed=None, model_name=""):
    
   # Generate a summary using RAG to retrieve relevant context first

    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # For summarization, we'll use the paper title as a query to get relevant parts
    # Extract a potential title from the first few lines
    first_lines = paper_content.split('\n', 10)[:10]
    potential_title = ""
    for line in first_lines:
        if len(line.strip()) > 10 and len(line.strip()) < 200:  # A reasonable title length
            potential_title = line.strip()
            break
    
    if not potential_title:
        potential_title = "paper summary"
    
    # Create query from potential title and generic summarization terms
    summary_query = f"{potential_title} main concepts methodology findings"
    
    # Get relevant context from RAG
    retrieved_context = rag_system.get_relevant_context(summary_query, top_k=5)
    
    # Create the system prompt
    system_prompt = """You are a helpful research assistant that summarizes academic papers. 
You should provide comprehensive, accurate summaries that capture the key points, methodologies, and findings of the paper."""

    # Create user prompt with the retrieved context
    user_prompt = f"""Please summarize the following documents:

PAPER CONTENT:
{retrieved_context}

Your summary should include:
1. The key concepts presented
2. Any practical applications or real-world examples
3. Include any historical context or notable scientists
4. The significance and implications of the concepts

Please be comprehensive yet concise and keep the summary to 300 words."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
"""
    
    # Log the prompt
    logger = logging.getLogger(__name__)
    logger.info(f"Generating summary using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1000,  # Longer for summaries
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

In [40]:
#Generate answer from LLM (no RAG version)
def generate_answer(model, tokenizer, paper_content, question, device, seed=None, model_name=""):
    """
    Generate an answer to a question based on paper content
    """
    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # Create the system prompt
    system_prompt = """You are a helpful research assistant that answers questions based on academic papers. 
You should provide concise, accurate answers based solely on the information in the paper context provided."""

    # Create user prompt with the paper content as context
    user_prompt = f"""Based on the following academic paper, please answer the question:

PAPER CONTENT:
{paper_content[:3000]}  # Limiting context size to avoid token limits

QUESTION:
{question}

Please provide a direct, concise answer using only information from the paper. If the paper doesn't contain relevant information to answer the question, please state that."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
"""
    
    # Log the prompt
    logger.info(f"Generating answer using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=500,  # Shorter for answers
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

def generate_summary(model, tokenizer, paper_content, device, seed=None, model_name=""):
    """
    Generate a summary of a paper
    """
    # Set seed for reproducibility if provided
    if seed is not None:
        torch.manual_seed(seed)
        random.seed(seed)
    
    # Create the system prompt
    system_prompt = """You are a helpful research assistant that summarizes academic papers. 
You should provide comprehensive, accurate summaries that capture the key points, methodologies, and findings of the paper."""

    # Create user prompt with the paper content as context
    user_prompt = f"""Please summarize the following documents:

PAPER CONTENT:
{paper_content[:3000]}  # Limiting context size to avoid token limits

Your summary should include:
1. The key concepts presented
2. Any practical applications or real-world examples
3. Include any historical context or notable scientists
4. The significance and implications of the concepts

Please be comprehensive yet concise and keep the summary to 300 words."""

    # Combine into chat format
    chat_prompt = f"""<|system|>
{system_prompt}

<|user|>
{user_prompt}

<|assistant|>
"""
    
    # Log the prompt
    logger.info(f"Generating summary using model: {model_name}")
    logger.info(f"Prompt length: {len(chat_prompt)}")
    
    # Tokenize
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    
    # Generate
    logger.info("Starting generation...")
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=1000,  # Longer for summaries
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode
    output = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Extract only the assistant's response
    assistant_response = output.split("<|assistant|>")[-1].strip()
    
    logger.info(f"Generation complete, response length: {len(assistant_response)}")
    return assistant_response

In [41]:
#Evaluate functions here
def evaluate_with_bertscore(generated_texts, reference_texts):
    """
    Evaluate generated texts using BERTScore
    """
    try:
        precision, recall, f1 = bert_score(
            generated_texts, 
            reference_texts, 
            lang="en", 
            verbose=True
        )
        return {
            "bertscore_precision": precision.mean().item(),
            "bertscore_recall": recall.mean().item(),
            "bertscore_f1": f1.mean().item()
        }
    except Exception as e:
        logger.error(f"Error calculating BERTScore: {str(e)}")
        return {
            "bertscore_precision": 0,
            "bertscore_recall": 0,
            "bertscore_f1": 0
        }

def evaluate_with_rouge(generated_texts, reference_texts):
    """
    Evaluate generated texts using ROUGE
    """
    try:
        rouge = Rouge()
        scores = rouge.get_scores(generated_texts, reference_texts, avg=True)
        return {
            "rouge1_f": scores["rouge-1"]["f"],
            "rouge2_f": scores["rouge-2"]["f"],
            "rougeL_f": scores["rouge-l"]["f"]
        }
    except Exception as e:
        logger.error(f"Error calculating ROUGE: {str(e)}")
        return {
            "rouge1_f": 0,
            "rouge2_f": 0,
            "rougeL_f": 0
        }

def setup_ragas_evaluation(use_open_llm=False, open_llm_name=None, openai_key=None, hf_token=None):
    """
    Setup RAGAS evaluation metrics and LLM
    """
    try:
        # Initialize metrics
        ragas_metrics = [
            context_precision,
            context_recall,
            faithfulness,
            answer_relevancy,
            answer_correctness
        ]
        
        if use_open_llm and open_llm_name:
            # Use open LLM via Hugging Face API
            if hf_token:
                from langchain_huggingface import HuggingFaceEndpoint
                
                # Create LangChain wrapper for the Hugging Face API endpoint
                ragas_llm = HuggingFaceEndpoint(
                    endpoint_url=f"https://api-inference.huggingface.co/models/{open_llm_name}",
                    huggingfacehub_api_token=hf_token,
                    task="text-generation",
                    max_new_tokens=512,
                    temperature=0.1,
                    model_kwargs={
                        "do_sample": False
                    }
                )
                logger.info(f"Using {open_llm_name} via Hugging Face API as judge")
            else:
                logger.warning("Hugging Face token not provided. Falling back to local model.")
                # Use local model pipeline
                from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline
                import torch
                from transformers import pipeline

                pipe = pipeline(
                    "text-generation",
                    model=open_llm_name,
                    torch_dtype=torch.bfloat16,
                    device_map="auto",
                    max_new_tokens=512,
                    do_sample=False
                )
                
                ragas_llm = HuggingFacePipeline(pipeline=pipe)
                logger.info(f"Using local model as judge: {open_llm_name}")
        else:
            # Use OpenAI model
            if openai_key:
                from langchain_openai import ChatOpenAI
                # Explicitly set the API key
                import os
                os.environ["OPENAI_API_KEY"] = openai_key
                ragas_llm = ChatOpenAI(model_name="gpt-3.5-turbo", openai_api_key=openai_key)
                logger.info("Using OpenAI GPT-3.5-Turbo as judge")
            else:
                logger.warning("OpenAI API key not provided. RAGAS evaluation will not be available.")
                return None, None
        
        return ragas_metrics, ragas_llm
    except Exception as e:
        logger.error(f"Error setting up RAGAS evaluation: {str(e)}")
        return None, None

In [42]:
#Run RAGAS here
def run_ragas_for_qa(paper_content, questions, generated_answers, reference_answers, ragas_metrics, ragas_llm):
    """
    Run RAGAS evaluation for QA task
    """
    try:
        # Prepare data for RAGAS
        # For Q&A, we need contexts, questions, answers, and references
        contexts = [[paper_content]] * len(questions)  # Wrap each in a list as RAGAS expects list of contexts
        
        eval_data = {
            "contexts": contexts,
            "question": questions,
            "answer": generated_answers,
            "reference": reference_answers
        }
        
        # Create Dataset
        dataset = Dataset.from_dict(eval_data)
        
        # Run evaluation
        result = ragas_evaluate(
            dataset=dataset,
            metrics=ragas_metrics,
            llm=ragas_llm
        )
        
        return result
    except Exception as e:
        logger.error(f"Error in RAGAS QA evaluation: {str(e)}")
        return None

def run_ragas_for_summary(paper_content, generated_summary, reference_summary, ragas_metrics, ragas_llm):
    """
    Run RAGAS evaluation for summary task
    """
    try:
        # For summaries, we create a dummy question asking for a summary
        question = ["Summarize the key points of this academic paper"]
        
        # Prepare data for RAGAS
        eval_data = {
            "contexts": [[paper_content]],  # Wrap in a list as RAGAS expects list of contexts
            "question": question,
            "answer": [generated_summary],
            "reference": [reference_summary]
        }
        
        # Create Dataset
        dataset = Dataset.from_dict(eval_data)
        
        # Run evaluation
        result = ragas_evaluate(
            dataset=dataset,
            metrics=ragas_metrics,
            llm=ragas_llm
        )
        
        return result
    except Exception as e:
        logger.error(f"Error in RAGAS summary evaluation: {str(e)}")
        return None

In [43]:
# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler("paper_lesson_generator.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Function to clear GPU/MPS memory
def clear_memory():
    gc.collect()
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        torch.mps.empty_cache()
    elif torch.cuda.is_available():
        torch.cuda.empty_cache()
    return True

In [44]:
# Helper class to manage multiple models
class ModelManager:
    def __init__(self, model_configs, use_gpu=True):
        """
        Initialize with multiple model configurations
        
        Args:
            model_configs: List of dictionaries with model_path and model_name keys
            use_gpu: Whether to use MPS or CUDA for acceleration
        """
        self.model_configs = model_configs
        self.use_gpu = use_gpu
        self.cuda_available = torch.cuda.is_available()
        self.mps_available = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

        if use_gpu:
            if torch.cuda.is_available():
                self.device = "cuda"
            elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
                self.device = "mps"
        else:
            self.device = "cpu"
        print(f"Using device: {use_gpu}")
        # Will hold loaded models and tokenizers
        self.models = {}
        self.tokenizers = {}
        
        logger.info(f"ModelManager initialized with device: {self.device}")
        logger.info(f"Models to load: {[cfg['model_name'] for cfg in model_configs]}")
    
    def load_model(self, model_key):
        """Load a specific model by its key in model_configs"""
        if model_key in self.models and model_key in self.tokenizers:
            logger.info(f"Model {model_key} already loaded")
            return self.models[model_key], self.tokenizers[model_key]
        
        # Find model config
        config = next((cfg for cfg in self.model_configs if cfg['model_name'] == model_key), None)
        if not config:
            raise ValueError(f"Model {model_key} not found in configurations")
        
        model_path = config['model_path']
        logger.info(f"Loading model: {model_key} from {model_path}")
        
        # Load tokenizer
        tokenizer = AutoTokenizer.from_pretrained(
            model_path, 
            use_fast=True, 
            local_files_only=True
        )
        
        # Setup quantization config for CUDA devices
        if self.device == "cuda":
            from transformers import BitsAndBytesConfig
        
            # Configure 8-bit quantization with CPU offloading enabled
            quantization_config = BitsAndBytesConfig(
                load_in_8bit=True,
                llm_int8_enable_fp32_cpu_offload=True,  # Enable CPU offloading for parts that don't fit in GPU
                llm_int8_skip_modules=None,             # Don't skip any modules for offloading
                llm_int8_threshold=6.0,                 # Default threshold
                bnb_4bit_compute_dtype=torch.float16    # Compute in float16
            )
        
            # Load model with optimized memory configuration and auto device mapping
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                device_map="auto",                    # Let the library determine optimal device mapping
                quantization_config=quantization_config,
                torch_dtype=torch.float16,
                low_cpu_mem_usage=True,
                local_files_only=True
            )
        
        # For MPS (Mac) devices or CPU
        elif self.device == "mps":
            model = AutoModelForCausalLM.from_pretrained(
            model_path,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
            local_files_only=True
        )
        # Safe to use .to() with MPS
        model = model.to(self.device)
    
        # CPU fallback
        else:
            model = AutoModelForCausalLM.from_pretrained(
                model_path,
                low_cpu_mem_usage=True,
            local_files_only=True
        )
        
        # Store loaded model and tokenizer
        self.models[model_key] = model
        self.tokenizers[model_key] = tokenizer
        
        logger.info(f"Model {model_key} loaded successfully")
        return model, tokenizer
    
    def unload_model(self, model_key):
        """Unload a model to free up memory"""
        if model_key in self.models:
            logger.info(f"Unloading model: {model_key}")
            del self.models[model_key]
            if model_key in self.tokenizers:
                del self.tokenizers[model_key]
            clear_memory()
            return True
        return False
    
    def unload_all_models(self):
        """Unload all models"""
        logger.info("Unloading all models")
        self.models.clear()
        self.tokenizers.clear()
        clear_memory()
        return True
    
    def get_model_names(self):
        """Return list of available model names"""
        return [cfg['model_name'] for cfg in self.model_configs]


In [45]:
def check_and_setup_api_keys():
    #Check if API keys are set and prompt user if not available"
    if os.getenv("COLAB_RELEASE_TAG"):
        from google.colab import userdata
        openai_key = os.environ.get("OPENAI_API_KEY")
        hf_token = os.environ.get("HUGGINGFACE_API_TOKEN")
    else:
        env_path = Path("C:/Users/luqma/AI6130/AI6130_Grp/AI6130.env")
        if env_path.exists():
            load_dotenv(env_path)
            openai_key = os.environ.get("OPENAI_API_KEY")
            print("OpenAI API key found in environment variables.")
            hf_token = os.environ.get("HUGGINGFACE_API_TOKEN")
            print("Hugging Face API token found in environment variables.")
            return openai_key, hf_token
        else:
            print("OpenAI API key not found in environment variables.")
            use_openai = input("Do you want to use OpenAI models for evaluation? (y/n): ").lower() == 'y'
            
            if use_openai:
                openai_key = input("Please enter your OpenAI API key: ")
                os.environ["OPENAI_API_KEY"] = openai_key
                print("OpenAI API key set for this session.")
            else:
                print("OpenAI evaluation will not be available.")
        
            print("Hugging Face API token not found in environment variables.")
            use_hf_api = input("Do you want to use Hugging Face API for evaluation? (y/n): ").lower() == 'y'

            if use_hf_api:
                hf_token = input("Please enter your Hugging Face API token: ")
                os.environ["HUGGINGFACE_API_TOKEN"] = hf_token
                print("Hugging Face API token set for this session.")
            else:
                print("Hugging Face API evaluation will not be available.")
    
    return openai_key , hf_token

In [46]:
def run_experiment_with_rag(model_configs, papers_dir="./papers", reference_dir="./reference", 
                  output_dir="./results", use_gpu=True, use_open_llm=False,
                  open_llm_name=None, openai_key=None, hf_token=None):
    """
    Run the full experiment with multiple PDFs, models, and evaluation metrics
    Using RAG for retrieving relevant passages
    """
    papers_dir = Path(papers_dir)
    reference_dir = Path(reference_dir)
    output_dir = Path(output_dir)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup logging
    experiment_log_path = os.path.join(output_dir, "experiment.log")
    file_handler = logging.FileHandler(experiment_log_path)
    file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
    logger.addHandler(file_handler)
    
    device = "cpu"
    if use_gpu:
        if torch.cuda.is_available():
            device = "cuda"
            logger.info(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
            logger.info("Using MPS (Metal Performance Shaders)")
        else:
            logger.info("No GPU available, using CPU")
    
    # Initialize model manager
    model_manager = ModelManager(model_configs, use_gpu=use_gpu)
    model_names = model_manager.get_model_names()
    
    # Load papers
    paper_contents = load_pdf_documents(papers_dir)
    if not paper_contents:
        logger.error("No paper contents loaded. Aborting experiment.")
        return
    
    # Initialize RAG system with all papers
    logger.info("Setting up RAG system...")
    rag_system = setup_rag_system(paper_contents)
    logger.info(f"RAG system initialized with {len(rag_system.documents)} chunks")
    
    # Load reference data
    qa_pairs_dict, reference_summaries = load_reference_data(reference_dir)
    
    # Setup RAGAS evaluation
    ragas_metrics, ragas_llm = setup_ragas_evaluation(use_open_llm, open_llm_name, openai_key, hf_token)
    if not ragas_metrics or not ragas_llm:
        logger.error("Failed to set up RAGAS evaluation. Continuing without RAGAS.")
    
    # Store all results
    all_results = {
        "qa_results": {},
        "summary_results": {},
        "bertscore_results": {},
        "rouge_results": {},
        "ragas_qa_results": {},
        "ragas_summary_results": {}
    }
    
    # Process each paper with each model
    for paper_file, paper_content in paper_contents.items():
        paper_basename = os.path.splitext(paper_file)[0]
        logger.info(f"Processing paper: {paper_file}")
        
        # Get QA pairs for this paper
        qa_pairs = qa_pairs_dict.get(paper_basename, [])
        if not qa_pairs:
            logger.warning(f"No QA pairs found for {paper_basename}. Skipping QA task.")
        
        # Get reference summary for this paper
        reference_summary = reference_summaries.get(paper_basename, "")
        if not reference_summary:
            logger.warning(f"No reference summary found for {paper_basename}. Skipping summary evaluation.")
        
        for model_name in model_names:
            logger.info(f"Processing with model: {model_name}")
            
            try:
                # Load model
                model, tokenizer = model_manager.load_model(model_name)
                
                # Generate answers to questions using RAG
                if qa_pairs:
                    questions = [pair["question"] for pair in qa_pairs]
                    reference_answers = [pair["answer"] for pair in qa_pairs]
                    
                    generated_answers = []
                    for i, question in enumerate(questions):
                        answer = generate_answer_with_rag(
                            model, tokenizer, rag_system, question, 
                            device, seed=hash(paper_file + question) % 10000, 
                            model_name=model_name
                        )
                        generated_answers.append(answer)
                        
                        # Save generated answer
                        qa_output_dir = os.path.join(output_dir, "qa", paper_basename, model_name)
                        os.makedirs(qa_output_dir, exist_ok=True)
                        with open(os.path.join(qa_output_dir, f"question_{i+1}.txt"), "w") as f:
                            f.write(f"QUESTION:\n{question}\n\nANSWER:\n{answer}")
                    
                    # Store QA results
                    all_results["qa_results"][(paper_basename, model_name)] = {
                        "questions": questions,
                        "generated_answers": generated_answers,
                        "reference_answers": reference_answers
                    }
                    
                    # Evaluate QA with BERTScore and ROUGE
                    bertscore_results = evaluate_with_bertscore(generated_answers, reference_answers)
                    rouge_results = evaluate_with_rouge(generated_answers, reference_answers)
                    
                    all_results["bertscore_results"][(paper_basename, model_name, "qa")] = bertscore_results
                    all_results["rouge_results"][(paper_basename, model_name, "qa")] = rouge_results
                    
                    # Run RAGAS for QA if available
                    if ragas_metrics and ragas_llm:
                        ragas_qa_result = run_ragas_for_qa(
                            paper_content, questions, generated_answers, 
                            reference_answers, ragas_metrics, ragas_llm
                        )
                        if ragas_qa_result:
                            all_results["ragas_qa_results"][(paper_basename, model_name)] = ragas_qa_result
                
                # Generate summary using RAG
                generated_summary = generate_summary_with_rag(
                    model, tokenizer, rag_system, paper_content, device, 
                    seed=hash(paper_file) % 10000, model_name=model_name
                )
                
                # Save generated summary
                summary_output_dir = os.path.join(output_dir, "summaries", paper_basename)
                os.makedirs(summary_output_dir, exist_ok=True)
                with open(os.path.join(summary_output_dir, f"{model_name}_summary.txt"), "w") as f:
                    f.write(generated_summary)
                
                # Store summary results
                all_results["summary_results"][(paper_basename, model_name)] = {
                    "generated_summary": generated_summary,
                    "reference_summary": reference_summary
                }
                
                # Evaluate summary with BERTScore and ROUGE if reference exists
                if reference_summary:
                    bertscore_results = evaluate_with_bertscore([generated_summary], [reference_summary])
                    rouge_results = evaluate_with_rouge(generated_summary, reference_summary)
                    
                    all_results["bertscore_results"][(paper_basename, model_name, "summary")] = bertscore_results
                    all_results["rouge_results"][(paper_basename, model_name, "summary")] = rouge_results
                    
                    # Run RAGAS for summary if available
                    if ragas_metrics and ragas_llm:
                        ragas_summary_result = run_ragas_for_summary(
                            paper_content, generated_summary, reference_summary, 
                            ragas_metrics, ragas_llm
                        )
                        if ragas_summary_result:
                            all_results["ragas_summary_results"][(paper_basename, model_name)] = ragas_summary_result
                
                # Unload model to free memory
                model_manager.unload_model(model_name)
                clear_memory()
                
            except Exception as e:
                logger.error(f"Error processing {paper_file} with model {model_name}: {str(e)}")
    
    # Save all results to CSV files
    save_results_to_csv(all_results, output_dir)
    
    # Generate summary report
    generate_report(all_results, output_dir)
    
    logger.info("Experiment completed successfully!")
    return all_results

In [47]:
def run_experiment(model_configs, papers_dir="./papers", reference_dir="./reference", 
                  output_dir="./results", use_gpu=True, use_open_llm=False,
                  open_llm_name=None, openai_key=None, hf_token=None):
    """
    Run the full experiment with multiple PDFs, models, and evaluation metrics
    """
    papers_dir = Path(papers_dir)
    reference_dir = Path(reference_dir)
    output_dir = Path(output_dir)
    
    # Create output directory
    os.makedirs(output_dir, exist_ok=True)
    
    # Setup logging
    experiment_log_path = os.path.join(output_dir, "experiment.log")
    file_handler = logging.FileHandler(experiment_log_path)
    file_handler.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
    logger.addHandler(file_handler)
    
    device = "cpu"
    if use_gpu:
        if torch.cuda.is_available():
            device = "cuda"
            logger.info(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            device = "mps"
            logger.info("Using MPS (Metal Performance Shaders)")
        else:
            logger.info("No GPU available, using CPU")
    
    # Initialize model manager
    model_manager = ModelManager(model_configs, use_gpu=use_gpu)
    model_names = model_manager.get_model_names()
    
    # Load papers
    paper_contents = load_pdf_documents(papers_dir)
    if not paper_contents:
        logger.error("No paper contents loaded. Aborting experiment.")
        return
    
    # Load reference data
    qa_pairs_dict, reference_summaries = load_reference_data(reference_dir)
    
    # Setup RAGAS evaluation
    ragas_metrics, ragas_llm = setup_ragas_evaluation(use_open_llm, open_llm_name, openai_key, hf_token)
    if not ragas_metrics or not ragas_llm:
        logger.error("Failed to set up RAGAS evaluation. Continuing without RAGAS.")
    
    # Store all results
    all_results = {
        "qa_results": {},
        "summary_results": {},
        "bertscore_results": {},
        "rouge_results": {},
        "ragas_qa_results": {},
        "ragas_summary_results": {}
    }
    
    # Process each paper with each model
    for paper_file, paper_content in paper_contents.items():
        paper_basename = os.path.splitext(paper_file)[0]
        logger.info(f"Processing paper: {paper_file}")
        
        # Get QA pairs for this paper
        qa_pairs = qa_pairs_dict.get(paper_basename, [])
        if not qa_pairs:
            logger.warning(f"No QA pairs found for {paper_basename}. Skipping QA task.")
        
        # Get reference summary for this paper
        reference_summary = reference_summaries.get(paper_basename, "")
        if not reference_summary:
            logger.warning(f"No reference summary found for {paper_basename}. Skipping summary evaluation.")
        
        for model_name in model_names:
            logger.info(f"Processing with model: {model_name}")
            
            try:
                # Load model
                model, tokenizer = model_manager.load_model(model_name)
                
                # Generate answers to questions
                if qa_pairs:
                    questions = [pair["question"] for pair in qa_pairs]
                    reference_answers = [pair["answer"] for pair in qa_pairs]
                    
                    generated_answers = []
                    for i, question in enumerate(questions):
                        answer = generate_answer(
                            model, tokenizer, paper_content, question, 
                            device, seed=hash(paper_file + question) % 10000, 
                            model_name=model_name
                        )
                        generated_answers.append(answer)
                        
                        # Save generated answer
                        qa_output_dir = os.path.join(output_dir, "qa", paper_basename, model_name)
                        os.makedirs(qa_output_dir, exist_ok=True)
                        with open(os.path.join(qa_output_dir, f"question_{i+1}.txt"), "w") as f:
                            f.write(f"QUESTION:\n{question}\n\nANSWER:\n{answer}")
                    
                    # Store QA results
                    all_results["qa_results"][(paper_basename, model_name)] = {
                        "questions": questions,
                        "generated_answers": generated_answers,
                        "reference_answers": reference_answers
                    }
                    
                    # Evaluate QA with BERTScore and ROUGE
                    bertscore_results = evaluate_with_bertscore(generated_answers, reference_answers)
                    rouge_results = evaluate_with_rouge(generated_answers, reference_answers)
                    
                    all_results["bertscore_results"][(paper_basename, model_name, "qa")] = bertscore_results
                    all_results["rouge_results"][(paper_basename, model_name, "qa")] = rouge_results
                    
                    # Run RAGAS for QA if available
                    if ragas_metrics and ragas_llm:
                        ragas_qa_result = run_ragas_for_qa(
                            paper_content, questions, generated_answers, 
                            reference_answers, ragas_metrics, ragas_llm
                        )
                        if ragas_qa_result:
                            all_results["ragas_qa_results"][(paper_basename, model_name)] = ragas_qa_result
                
                # Generate summary
                generated_summary = generate_summary(
                    model, tokenizer, paper_content, device, 
                    seed=hash(paper_file) % 10000, model_name=model_name
                )
                
                # Save generated summary
                summary_output_dir = os.path.join(output_dir, "summaries", paper_basename)
                os.makedirs(summary_output_dir, exist_ok=True)
                with open(os.path.join(summary_output_dir, f"{model_name}_summary.txt"), "w") as f:
                    f.write(generated_summary)
                
                # Store summary results
                all_results["summary_results"][(paper_basename, model_name)] = {
                    "generated_summary": generated_summary,
                    "reference_summary": reference_summary
                }
                
                # Evaluate summary with BERTScore and ROUGE if reference exists
                if reference_summary:
                    bertscore_results = evaluate_with_bertscore([generated_summary], [reference_summary])
                    rouge_results = evaluate_with_rouge(generated_summary, reference_summary)
                    
                    all_results["bertscore_results"][(paper_basename, model_name, "summary")] = bertscore_results
                    all_results["rouge_results"][(paper_basename, model_name, "summary")] = rouge_results
                    
                    # Run RAGAS for summary if available
                    if ragas_metrics and ragas_llm:
                        ragas_summary_result = run_ragas_for_summary(
                            paper_content, generated_summary, reference_summary, 
                            ragas_metrics, ragas_llm
                        )
                        if ragas_summary_result:
                            all_results["ragas_summary_results"][(paper_basename, model_name)] = ragas_summary_result
                
                # Unload model to free memory
                model_manager.unload_model(model_name)
                clear_memory()
                
            except Exception as e:
                logger.error(f"Error processing {paper_file} with model {model_name}: {str(e)}")
    
    # Save all results to CSV files
    save_results_to_csv(all_results, output_dir)
    
    # Generate summary report
    generate_report(all_results, output_dir)
    
    logger.info("Experiment completed successfully!")
    return all_results

In [48]:
def save_results_to_csv(all_results, output_dir):
    """
    Save experiment results to CSV files
    """
    # Save BERTScore results
    bertscore_data = []
    for (paper, model, task), scores in all_results["bertscore_results"].items():
        row = {
            "paper": paper,
            "model": model,
            "task": task,
            **scores
        }
        bertscore_data.append(row)
    
    if bertscore_data:
        bertscore_df = pd.DataFrame(bertscore_data)
        bertscore_df.to_csv(os.path.join(output_dir, "bertscore_results.csv"), index=False)
    
    # Save ROUGE results
    rouge_data = []
    for (paper, model, task), scores in all_results["rouge_results"].items():
        row = {
            "paper": paper,
            "model": model,
            "task": task,
            **scores
        }
        rouge_data.append(row)
    
    if rouge_data:
        rouge_df = pd.DataFrame(rouge_data)
        rouge_df.to_csv(os.path.join(output_dir, "rouge_results.csv"), index=False)
    
    # Save RAGAS QA results
    if all_results["ragas_qa_results"]:
        ragas_qa_dfs = []
        for (paper, model), result in all_results["ragas_qa_results"].items():
            df = result.to_pandas()
            df["paper"] = paper
            df["model"] = model
            ragas_qa_dfs.append(df)
        
        ragas_qa_df = pd.concat(ragas_qa_dfs, ignore_index=True)
        ragas_qa_df.to_csv(os.path.join(output_dir, "ragas_qa_results.csv"), index=False)
    
    # Save RAGAS summary results
    if all_results["ragas_summary_results"]:
        ragas_summary_dfs = []
        for (paper, model), result in all_results["ragas_summary_results"].items():
            df = result.to_pandas()
            df["paper"] = paper
            df["model"] = model
            ragas_summary_dfs.append(df)
        
        ragas_summary_df = pd.concat(ragas_summary_dfs, ignore_index=True)
        ragas_summary_df.to_csv(os.path.join(output_dir, "ragas_summary_results.csv"), index=False)

def generate_report(all_results, output_dir):
    """
    Generate a markdown report summarizing the experiment results
    """
    report_path = os.path.join(output_dir, "experiment_report.md")
    
    with open(report_path, "w") as f:
        f.write("# RAG Experiment Report\n\n")
        f.write(f"Date: {time.strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        # Write summary of experiment
        papers = set([paper for (paper, _) in all_results["summary_results"].keys()])
        models = set([model for (_, model) in all_results["summary_results"].keys()])
        
        f.write(f"## Experiment Overview\n\n")
        f.write(f"- Number of papers: {len(papers)}\n")
        f.write(f"- Models evaluated: {', '.join(models)}\n")
        f.write(f"- Tasks: Question Answering and Summarization\n\n")
        
        # Write BERTScore results
        f.write("## BERTScore Results\n\n")
        
        # Summaries
        f.write("### Summaries\n\n")
        f.write("| Paper | Model | Precision | Recall | F1 |\n")
        f.write("|-------|-------|-----------|--------|----|\n")
        
        for (paper, model, task) in all_results["bertscore_results"]:
            if task == "summary":
                scores = all_results["bertscore_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['bertscore_precision']:.4f} | {scores['bertscore_recall']:.4f} | {scores['bertscore_f1']:.4f} |\n")
        
        # QA
        f.write("\n### Question Answering\n\n")
        f.write("| Paper | Model | Precision | Recall | F1 |\n")
        f.write("|-------|-------|-----------|--------|----|\n")
        
        for (paper, model, task) in all_results["bertscore_results"]:
            if task == "qa":
                scores = all_results["bertscore_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['bertscore_precision']:.4f} | {scores['bertscore_recall']:.4f} | {scores['bertscore_f1']:.4f} |\n")
        
        # Write ROUGE results
        f.write("\n## ROUGE Results\n\n")
        
        # Summaries
        f.write("### Summaries\n\n")
        f.write("| Paper | Model | ROUGE-1 | ROUGE-2 | ROUGE-L |\n")
        f.write("|-------|-------|---------|---------|--------|\n")
        
        for (paper, model, task) in all_results["rouge_results"]:
            if task == "summary":
                scores = all_results["rouge_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['rouge1_f']:.4f} | {scores['rouge2_f']:.4f} | {scores['rougeL_f']:.4f} |\n")
        
        # QA
        f.write("\n### Question Answering\n\n")
        f.write("| Paper | Model | ROUGE-1 | ROUGE-2 | ROUGE-L |\n")
        f.write("|-------|-------|---------|---------|--------|\n")
        
        for (paper, model, task) in all_results["rouge_results"]:
            if task == "qa":
                scores = all_results["rouge_results"][(paper, model, task)]
                f.write(f"| {paper} | {model} | {scores['rouge1_f']:.4f} | {scores['rouge2_f']:.4f} | {scores['rougeL_f']:.4f} |\n")
        
        # RAGAS results
        if all_results["ragas_qa_results"]:
            f.write("\n## RAGAS QA Results\n\n")
            f.write("RAGAS results are available in the CSV files.\n\n")
        
        if all_results["ragas_summary_results"]:
            f.write("\n## RAGAS Summary Results\n\n")
            f.write("RAGAS summary results are available in the CSV files.\n\n")
        
        f.write("\n\n*Note: Full results are available in the CSV files in the output directory.*\n")
    
    logger.info(f"Report generated at {report_path}")

In [52]:
def setConfigparam():
    # Check and setup API keys
    openai_key, hf_token = check_and_setup_api_keys()

    if torch.cuda.is_available():
        device = "cuda"
        print(f"Using CUDA device: {torch.cuda.get_device_name(0)}")
        gpu_available = True
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = "mps"
        print("Using MPS (Metal Performance Shaders)")
        gpu_available = True
    else:
        print(f"MPS (Metal Performance Shaders) acceleration: {'Not available'}")
        gpu_available = False

    path_name = "C:/Users/luqma/AI6130/AI6130_Grp/"
    papers_dir = Path(path_name + "papers")
    reference_dir = Path(path_name + "reference")
    output_dir = Path(path_name + "results")
    model_dir = Path(path_name + "models")

    return openai_key,hf_token, gpu_available, papers_dir, reference_dir,output_dir, model_dir


In [65]:
if __name__ == "__main__":
    print("RAG-Enabled Paper Evaluation: QA and Summarization for Multiple Papers and Models")
    print("=" * 80)

    #set parameters
    openai_key,hf_token, gpu_available, papers_dir, reference_dir,output_dir, model_dir = setConfigparam()
    
    print( f" Using: {gpu_available}") 
    
    print("\nOptions:")
    print("1. Run with default models (RAG-enabled)")
    print("2. Configure custom models (RAG-enabled)")
    print("3. Run with default models + open LLM as judge (RAG-enabled)")
    print("4. Run RAG demonstration only")
    print("5. Run with default models (without RAG)")
    
    try:
        choice = input("\nSelect option (1-5): ")
        
        if choice == "1":
            # Default configuration with RAG
            model_configs = [
                {
                    "model_name": "meta-llama/Llama-3.2-3B",
                    "model_path": "C:/Users/luqma/AI6130/AI6130_Grp/Models/llama-3.2-3B"
                },
                {
                    "model_name": "google/Gemma-3-4B-it",
                    "model_path":  "C:/Users/luqma/AI6130/AI6130_Grp/Models/gemma-3-4B-it"
                }
            ]
            
            print("\nRunning with RAG-enabled default models...")
            results = run_experiment_with_rag(
                model_configs=model_configs,
                papers_dir=paper_dir,
                reference_dir=reference_dir,
                output_dir=output_dir,
                use_gpu=gpu_available,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        elif choice == "2":
            # Custom model configuration with RAG
            model_configs = []
            
            print("\nEnter details for Model 1:")
            model1_name = input("Model 1 name (e.g., Llama-3.2-3B): ")
            model1_path = input("Model 1 path (local or HF): ")
            model_configs.append({
                "model_name": model1_name,
                "model_path": model1_path
            })
            
            #print("\nEnter details for Model 2:")
            #model2_name = input("Model 2 name (e.g., Gemma-3-4B-it): ")
            # model2_path = input("Model 2 path (local or HF): ")
            
            #model_configs.append({
            #    "model_name": model2_name,
            #    "model_path": model2_path
            #})
            
            # Optional third model
            #add_third = input("\nAdd a third model? (y/n): ").lower()
            #if add_third == 'y':
            #    print("\nEnter details for Model 3:")
            #    model3_name = input("Model 3 name: ")
            #    model3_path = input("Model 3 path (local or HF): ")
                
            #    model_configs.append({
            #        "model_name": model3_name,
            #        "model_path": model3_path
            #    })
            
            #papers_dir = input("\nEnter path to papers directory (default: ./papers): ") or "./papers"
           # reference_dir = input("Enter path to reference directory (default: ./reference): ") or "./reference"
            ############output_dir = input("Enter path to output directory (default: ./results_rag): ") or "./results_rag"
            
            print("\nRunning with RAG-enabled custom models...")
            results = run_experiment_with_rag(
                model_configs=model_configs,
                papers_dir=papers_dir,
                reference_dir=reference_dir,
                output_dir=output_dir,
                use_gpu=gpu_available,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        elif choice == "3":
            # Default models with open LLM as judge and RAG
            model_configs = [
                {
                    "model_name": "Llama-3.2-3B",
                    "model_path": Path(model_dir + "/llama-3.2-3B")
                },
                {
                    "model_name": "Gemma-3-4B-it",
                    "model_path": Path(model_dir + "/gemma-3-4B-it")
                }
            ]
            
            open_llm_name = input("\nEnter open LLM model name for evaluation (default: mistralai/Mistral-7B-v0.1): ") or "mistralai/Mistral-7B-v0.1"
            
            print(f"\nRunning with RAG-enabled default models and {open_llm_name} as judge...")
            results = run_experiment_with_rag(
                model_configs=model_configs,
                papers_dir=paper_dir,
                reference_dir=reference_dir,
                output_dir=output_dir,
                use_gpu=gpu_available,
                use_open_llm=True,
                open_llm_name=open_llm_name,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        elif choice == "4":
            # Run only the RAG demonstration
            print("\nRunning RAG demonstration...")
            example_rag_usage()
            
        elif choice == "5":
            # Default configuration without RAG
            model_configs = [
                {
                    "model_name": "c",
                    "model_path": "meta-llama/llama-3.2-3B"
                },
                {
                    "model_name": "Gemma-3-4B-it",
                    "model_path": "google/gemma-3-4B-it"
                }
            ]
            
            print("\nRunning with default models (no RAG)...")
            results = run_experiment(  # Using the original non-RAG function
                model_configs=model_configs,
                papers_dir=paper_dir,
                reference_dir=reference_dir,
                output_dir=output_dir,
                use_gpu=gpu_available,
                openai_key=openai_key,
                hf_token=hf_token
            )
            
        else:
            print("Invalid choice.")
            
    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

RAG-Enabled Paper Evaluation: QA and Summarization for Multiple Papers and Models
OpenAI API key found in environment variables.
Hugging Face API token found in environment variables.
Using CUDA device: NVIDIA GeForce RTX 4070 Ti
 Using: True

Options:
1. Run with default models (RAG-enabled)
2. Configure custom models (RAG-enabled)
3. Run with default models + open LLM as judge (RAG-enabled)
4. Run RAG demonstration only
5. Run with default models (without RAG)



Select option (1-5):  2



Enter details for Model 1:


Model 1 name (e.g., Llama-3.2-3B):  llama-3.2-3B
Model 1 path (local or HF):  C:/Users/luqma/AI6130/AI6130_Grp/Models/llama-3.2-3B


2025-04-11 01:28:28,369 [INFO] Using CUDA device: NVIDIA GeForce RTX 4070 Ti
2025-04-11 01:28:28,369 [INFO] ModelManager initialized with device: cuda
2025-04-11 01:28:28,370 [INFO] Models to load: ['llama-3.2-3B']
2025-04-11 01:28:28,371 [INFO] Loading document: C:\Users\luqma\AI6130\AI6130_Grp\papers\Force.pdf
2025-04-11 01:28:28,371 [INFO] Loading document: C:\Users\luqma\AI6130\AI6130_Grp\papers\Force.pdf
2025-04-11 01:28:28,425 [INFO] Successfully loaded Force.pdf (14631 characters)
2025-04-11 01:28:28,426 [INFO] Loading document: C:\Users\luqma\AI6130\AI6130_Grp\papers\Introduction_to_Dynamics.pdf
2025-04-11 01:28:28,426 [INFO] Loading document: C:\Users\luqma\AI6130\AI6130_Grp\papers\Introduction_to_Dynamics.pdf
2025-04-11 01:28:28,564 [INFO] Successfully loaded Introduction_to_Dynamics.pdf (15516 characters)


...

2025-04-11 01:28:31,703 [INFO] Updated FAISS index with 7 chunks
2025-04-11 01:28:31,703 [INFO] Adding document to RAG system: Introduction_to_Dynamics.pdf
2025-04-11 01:28:31,704 [INFO] Document chunked into 8 parts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-04-11 01:28:31,827 [INFO] Updated FAISS index with 15 chunks
2025-04-11 01:28:31,827 [INFO] Adding document to RAG system: Newtons_Law.pdf
2025-04-11 01:28:31,829 [INFO] Document chunked into 7 parts


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2025-04-11 01:28:31,988 [INFO] Updated FAISS index with 22 chunks
2025-04-11 01:28:31,989 [INFO] RAG system initialized with 22 chunks from 3 documents
2025-04-11 01:28:31,989 [INFO] RAG system initialized with 22 chunks
2025-04-11 01:28:31,992 [INFO] Loaded QA pairs for 3 papers
2025-04-11 01:28:31,993 [INFO] Loaded summary for Force
2025-04-11 01:28:31,993 [INFO] Loaded summary for Introduction_to_Dynamics
2025-04-11 01:28:31,994 [INFO] Loaded summary for Newtons_Laws
2025-04-11 01:28:32,006 [INFO] Using OpenAI GPT-3.5-Turbo as judge
2025-04-11 01:28:32,006 [INFO] Processing paper: Force.pdf
2025-04-11 01:28:32,008 [INFO] Processing with model: llama-3.2-3B
2025-04-11 01:28:32,008 [INFO] Loading model: llama-3.2-3B from C:/Users/luqma/AI6130/AI6130_Grp/Models/llama-3.2-3B
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
2025-04-11 01:28:32,27

In [ ]:
#Example usage
def example_usage():

    #getAPI key
    openai_key, hf_token = check_and_setup_api_keys()
    # Define model configurations
    model_configs = [
        {
            "model_name": "Llama3-8B",
            "model_path": "./models/Llama-3.1-8B"
        },
        {
            "model_name": "DeepSeek-R1-Distill-Llama-8B",
            "model_path": "./models/DeepSeek-R1-Distill-Llama-8B"
        }
    ]
    
    # Set paths
    papers_dir = "./papers"
    reference_dir = "./reference"
    output_dir = "./experiment_results"
    
    # Run experiment
    results = run_experiment(
        model_configs=model_configs,
        papers_dir=papers_dir,
        reference_dir=reference_dir,
        output_dir=output_dir,
        use_gpu=True,
        use_open_llm=False  # Set to True to use open LLM as judge
        openai_key=openai_key
        hf_token=hf_token
    )
    
    return results

# Example with open LLM as judge
def example_with_open_llm():
    # Define model configurations
    model_configs = [
        {
            "model_name": "Llama3-8B",
            "model_path": "./models/Llama-3.1-8B"
        }
    ]
    
    # Set paths
    papers_dir = "./papers"
    reference_dir = "./reference"
    output_dir = "./experiment_results_open_llm"
    
    # Run experiment with open LLM
    results = run_experiment(
        model_configs=model_configs,
        papers_dir=papers_dir,
        reference_dir=reference_dir,
        output_dir=output_dir,
        use_gpu=True,
        use_open_llm=True,
        open_llm_name="mistralai/Mistral-7B-v0.1"
    )